# MFT and FFT propagation: Fraunhofer and Fresnel

This notebook makes the 2 × 2 propagation contract executable. **MFT versus FFT selects the numerical transform; Fraunhofer versus Fresnel selects the physical regime.** Fraunhofer is the default regime for both methods. The compact reference functions below will be replaced by the public implementation.

In [ ]:
import matplotlib.pyplot as plt
import torch

from fiatlux.core import Grid

torch.set_default_dtype(torch.float64)

wavelength = 632.8e-9
propagation_scale = 0.20  # focal length for Fraunhofer, distance for Fresnel
n = 65  # odd sampling keeps the centred spatial and FFT origins aligned
dx = 20e-6
input_grid = Grid(nx=n, ny=n, dx=dx, dy=dx, dtype=torch.float64)
output_dx = wavelength * propagation_scale / (n * dx)
output_grid = Grid(
    nx=n, ny=n, dx=output_dx, dy=output_dx, dtype=torch.float64
)

x, y = input_grid.meshgrid()
r2 = x.square() + y.square()
waist = 0.18e-3
input_field = torch.exp(-r2 / waist**2).to(torch.complex128)
input_field /= (
    input_field.abs().square().sum() * input_grid.dx * input_grid.dy
).sqrt()

print(f"input sampling:  {input_grid.dx * 1e6:.2f} µm")
print(f"natural output sampling: {output_grid.dx * 1e6:.2f} µm")

## Numerical transforms

The MFT evaluates the Fourier integral on explicit coordinates. The FFT evaluates it on its natural conjugate grid, whose sampling obeys \(dx_2=\lambda q/(n_x dx_1)\). On that common grid the complex results must agree.

In [ ]:
def mft2(field, input_grid, output_grid, wavelength, scale):
    mx = torch.exp(
        -2j * torch.pi * torch.outer(
            input_grid.x, output_grid.x / (wavelength * scale)
        )
    )
    my = torch.exp(
        -2j * torch.pi * torch.outer(
            input_grid.y, output_grid.y / (wavelength * scale)
        )
    )
    return (
        my.T @ field @ mx
        * input_grid.dx * input_grid.dy
        / (wavelength * scale)
    )


def fft2(field, input_grid, wavelength, scale):
    return (
        torch.fft.fftshift(
            torch.fft.fft2(torch.fft.ifftshift(field), norm="backward")
        )
        * input_grid.dx * input_grid.dy
        / (wavelength * scale)
    )

## The four combinations

Fraunhofer transforms the input field directly. Fresnel applies the same MFT or FFT to the input field multiplied by a quadratic phase, then applies the output quadratic phase and carrier.

In [ ]:
def propagate(field, method, regime):
    transform = mft2 if method == "mft" else fft2

    if regime == "fraunhofer":
        if method == "mft":
            return transform(
                field, input_grid, output_grid, wavelength, propagation_scale
            )
        return transform(field, input_grid, wavelength, propagation_scale)

    if regime != "fresnel":
        raise ValueError(f"unknown propagation regime: {regime}")

    k = 2 * torch.pi / wavelength
    input_chirp = torch.exp(1j * k * r2 / (2 * propagation_scale))
    x2, y2 = output_grid.meshgrid()
    output_chirp = torch.exp(
        1j * k * (x2.square() + y2.square()) / (2 * propagation_scale)
    )
    if method == "mft":
        transformed = transform(
            field * input_chirp,
            input_grid,
            output_grid,
            wavelength,
            propagation_scale,
        )
    else:
        transformed = transform(
            field * input_chirp, input_grid, wavelength, propagation_scale
        )
    carrier = torch.exp(
        torch.as_tensor(1j * k * propagation_scale, dtype=field.dtype)
    )
    return carrier / 1j * output_chirp * transformed


fraunhofer_mft = propagate(input_field, "mft", "fraunhofer")
fraunhofer_fft = propagate(input_field, "fft", "fraunhofer")
fresnel_mft = propagate(input_field, "mft", "fresnel")
fresnel_fft = propagate(input_field, "fft", "fresnel")

## Quantitative checks

For matching natural sampling, MFT and FFT must agree as complex fields in both regimes. All four propagated fields must preserve integrated flux.

In [ ]:
torch.testing.assert_close(fraunhofer_mft, fraunhofer_fft, rtol=1e-10, atol=1e-10)
torch.testing.assert_close(fresnel_mft, fresnel_fft, rtol=1e-10, atol=1e-10)

def flux(field, grid):
    return field.abs().square().sum() * grid.dx * grid.dy

input_flux = flux(input_field, input_grid)
for name, field in {
    "MFT Fraunhofer": fraunhofer_mft,
    "FFT Fraunhofer": fraunhofer_fft,
    "MFT Fresnel": fresnel_mft,
    "FFT Fresnel": fresnel_fft,
}.items():
    output_flux = flux(field, output_grid)
    torch.testing.assert_close(output_flux, input_flux, rtol=1e-10, atol=1e-12)
    print(f"{name:17s}: flux = {float(output_flux):.12f}")

## Visual validation

In [ ]:
fields = [
    fraunhofer_mft, fraunhofer_fft, fresnel_mft, fresnel_fft
]
titles = [
    "MFT · Fraunhofer", "FFT · Fraunhofer",
    "MFT · Fresnel", "FFT · Fresnel",
]
extent_mm = [
    float(output_grid.x.min() * 1e3), float(output_grid.x.max() * 1e3),
    float(output_grid.y.min() * 1e3), float(output_grid.y.max() * 1e3),
]
fig, axes = plt.subplots(2, 2, figsize=(9, 8), constrained_layout=True)
for ax, field, title in zip(axes.flat, fields, titles):
    image = ax.imshow(
        field.abs().square().detach().cpu().numpy(),
        origin="lower",
        extent=extent_mm,
    )
    ax.set_title(title)
    ax.set_xlabel("x [mm]")
    ax.set_ylabel("y [mm]")
    fig.colorbar(image, ax=ax)
plt.show()

The production API must preserve this 2 × 2 structure. Fraunhofer remains the default physical regime for both transform methods; choosing MFT or FFT changes sampling freedom and computational strategy, not the propagation physics.